In [116]:
import numpy as np
import os
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

In [117]:
def load_data(folder):
    x_train = np.load(os.path.join(folder, 'x_train.npy'))
    y_train = np.load(os.path.join(folder, 'y_train.npy'))
    x_test = np.load(os.path.join(folder, 'x_test.npy'))
    y_test = np.load(os.path.join(folder, 'y_test.npy'))
    return x_train, y_train, x_test, y_test

In [118]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


class LogisticRegression:
    def __init__(self, dim=2):
        rng = np.random.default_rng(seed=0)
        self.w = rng.normal(size=(dim, 1)) / np.sqrt(dim)
        self.b = np.zeros((1,))

    def predict(self, x, probs=False):
        # x - np.array размерности [N, dim]
        #     Массив входных признаков.
        assert x.shape[1] == self.w.shape[0], \
            "Размерность экземпляров данных не соответствует ожидаемой: " + \
            f"ожидалось x.shape[1]={self.w.shape[0]}, но было получено x.shape[1]={x.shape[1]}"

        x = x.dot(self.w) + self.b  # logits
        p = sigmoid(x)  # probabilities
        if probs:
            return p
        return np.array(p > 0.5).astype('int32')

    def fit(self, x, y, iters=1000, lr=0.01):
        # x - np.array размерности [N, dim]
        #     Массив входных признаков.
        # y - np.array размернсоти [N]
        #     Массив меток (правильных ответов).
        assert len(x) == len(y), \
            "Количество экземпляров в массиве X не равно количеству меток в массиве Y. " + \
            f"Полученные размеры: len(X) = {len(x)}, len(Y) = {len(y)}."
        assert x.shape[1] == self.w.shape[0], \
            "Размерность экземпляров данных не соответствует ожидаемой: " + \
            f"ожидалось x.shape[1]={self.w.shape[0]}, но было получено x.shape[1]={x.shape[1]}"
        # Алгоритм градиентного спуска.
        # Минимизируется бинарная кросс-энтропия.
        y = y.reshape(-1, 1)
        for i in range(iters):
            preds = self.predict(x, probs=True)
            self.w -= lr * np.mean(x.T.dot(preds - y), axis=1, keepdims=True)
            self.b -= lr * np.mean(preds - y, axis=0)
        return self

## 1. Применение логистической регрессии (несбалансированные данные)

### 1.1 Создание и обучение логистической регрессии

In [119]:
# Указание: производить нормализацию данных не нужно, это часть задания.
x_train, y_train, x_test, y_test = load_data('dataset1')

In [120]:
# Создайте модель логистической регрессии и обучите её, используя метод fit.
model = LogisticRegression(dim=x_train.shape[1])
model.fit(x_train, y_train)
print("Модель обучена")

Модель обучена


In [121]:
# Получите предсказания на тестовой выборке и оцените точность модели,
# используя accuracy_score из пакета SciKit-Learn.
y_pred = model.predict(x_test)
print(f"Accuracy модели логистической регрессии: {accuracy_score(y_test, y_pred):.4f}")

# Анализ:
# Классификатор демонстрирует высокую точность, но для полной оценки модели требуется анализ дополнительных метрик.

Accuracy модели логистической регрессии: 0.9045


### 1.2 Анализ качества модели

In [122]:
# Допишите класс "глупого классификатора", что всегда предсказывает класс `0`.
class DummyClassifier:
    def __init__(self):
        print('Hello, brother!')

    def predict(self, x):
        # x - numpy массив размерности [N, dim]
        # Должен возвращаться массив N предсказаний
        return np.zeros(len(x), dtype=int) # Всегда предсказываем класс 0

In [123]:
# Оцените точность "глупого классификатора", объясните результат.
dummy_model = DummyClassifier()
y_pred_dummy = dummy_model.predict(x_test)
print(f"Accuracy DummyClassifier: {accuracy_score(y_test, y_pred_dummy):.4f}")

# Анализ:
# Несмотря на высокую общую точность, модель на самом деле не справляется с задачей, поскольку данные несбалансированы.
# Классификатор просто "угадывает" преобладающий класс 0, что создает ложное впечатление его эффективности.

Hello, brother!
Accuracy DummyClassifier: 0.9091


In [124]:
# Используйте дополнительные метрики (f1-score, recall, precision) из пакета sklearn для анализа "глупого классификатора".
print('Дополнительные метрики "глупого классификатора":')
print("F1-score :", f"{f1_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print("Recall   :", f"{recall_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print("Precision:", f"{precision_score(y_test, y_pred_dummy, zero_division=0):.4f}")

# Анализ:
# Отсутствие правильных предсказаний для класса 1 привело к обнулению всех метрик.

Дополнительные метрики "глупого классификатора":
F1-score : 0.0000
Recall   : 0.0000
Precision: 0.0000


In [125]:
# Используя те же метрики, проанализируйте обученную вами модель логистической регрессии.
y_pred_lr = model.predict(x_test)
print(f"\nМетрики модели логистической регрессии:")
print("F1-score :", f"{f1_score(y_test, y_pred_lr):.4f}")
print("Recall   :", f"{recall_score(y_test, y_pred_lr):.4f}")
print("Precision:", f"{precision_score(y_test, y_pred_lr):.4f}")

# Анализ:
# Модель демонстрирует способность к распознаванию класса 1 по сравнению с "глупым классификатором", даже при условии получения невысоких значений.


Метрики модели логистической регрессии:
F1-score : 0.4000
Recall   : 0.3500
Precision: 0.4667


In [ ]:
# Объясните результат, описав его комментариями в этой клетке.

# Анализ:
# Результаты эксперимента подчеркивают, что высокая точность модели может быть обманчивой на несбалансированных данных.
# Простой "глупый классификатор", предсказывающий только класс 0, достиг высокой общей точности, но показал нулевые F1-score, recall и precision,
# что свидетельствует о полной неспособности обнаружить объекты класса 1.
# Такая модель просто запоминает доминирующий класс, выдавая однообразные предсказания, что создает иллюзию точности, но делает ее непригодной для реальных задач.

# Логистическая регрессия, показав чуть более низкую точность, тем не менее, смогла корректно выделить часть объектов меньшинства.
# Это указывает на ее способность к распознаванию обоих классов, пусть и с переменным успехом.

# Следовательно, логистическая регрессия является более эффективной моделью, поскольку она отражает структуру данных и обнаруживает примеры обоих классов,
# в отличие от "глупого классификатора", который создает лишь видимость высокой точности за счет перекоса в данных.

### 1.3 Анализ набора данных

In [126]:
print(f"Количество экземпляров в обучающей выборке по классам: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Количество экземпляров в тестовой выборке по классам: {dict(zip(*np.unique(y_test, return_counts=True)))}")

Количество экземпляров в обучающей выборке по классам: {np.float32(0.0): np.int64(200), np.float32(1.0): np.int64(20)}
Количество экземпляров в тестовой выборке по классам: {np.float32(0.0): np.int64(200), np.float32(1.0): np.int64(20)}


In [127]:
# Предложите способ улучшения качества модели. Подсказка: добавление дубликатов в данные.
# Указание: не изменяйте тестовую выборку.

# Способ улучшения: Сбалансировать обучающую выборку, добавив дубликаты экземпляров миноритарного класса (класса 1).
# Это позволит модели лучше изучить признаки этого класса.

# Отделяем данные по классам
x_train_0 = x_train[y_train == 0]
y_train_0 = y_train[y_train == 0]
x_train_1 = x_train[y_train == 1]
y_train_1 = y_train[y_train == 1]

# Определяем, сколько экземпляров класса 0 и класса 1
num_class_0 = len(x_train_0)
num_class_1 = len(x_train_1)

# Если класс 1 является миноритарным, дублируем его экземпляры, чтобы он соответствовал количеству класса 0
if num_class_1 < num_class_0:
    # Вычисляем, сколько раз нужно продублировать класс 1
    num_duplicates = num_class_0 // num_class_1

    # Создаем новые массивы X и Y для класса 1 путем конкатенации
    x_train_1_balanced = np.tile(x_train_1, (num_duplicates, 1))
    y_train_1_balanced = np.tile(y_train_1, num_duplicates)

    # Объединяем сбалансированные данные класса 1 с данными класса 0
    x_train_balanced = np.concatenate((x_train_0, x_train_1_balanced), axis=0)
    y_train_balanced = np.concatenate((y_train_0, y_train_1_balanced), axis=0)

    # Перемешиваем сбалансированные данные, чтобы избежать упорядоченности
    shuffle_indices = np.arange(len(x_train_balanced))
    np.random.shuffle(shuffle_indices)
    x_train_balanced = x_train_balanced[shuffle_indices]
    y_train_balanced = y_train_balanced[shuffle_indices]

    print(f"\nСбалансированная обучающая выборка создана")
    unique_balanced, counts_balanced = np.unique(y_train_balanced, return_counts=True)
    class_counts_balanced = dict(zip(unique_balanced, counts_balanced))
    print(f"Количество экземпляров в сбалансированной обучающей выборке по классам: {class_counts_balanced}")



Сбалансированная обучающая выборка создана
Количество экземпляров в сбалансированной обучающей выборке по классам: {np.float32(0.0): np.int64(200), np.float32(1.0): np.int64(200)}


In [128]:
# Создайте и обучите модель с использованием предложенных наработок.
model_balanced = LogisticRegression(dim=x_train_balanced.shape[1])
model_balanced.fit(x_train_balanced, y_train_balanced)
print("Модель обучена")

Модель обучена


In [129]:
# Оцените качество новой модели, используя метрики из пакета sklearn.metrics.
# Указание: постарайтесь сбалансировать данные таким образом, чтобы новая модель была ощутимо лучше старой.

y_pred_balanced = model_balanced.predict(x_test)

print(f"\nМетрики модели логистической регрессии (сбалансированные данные):")
print("Accuracy :", f"{accuracy_score(y_test, y_pred_balanced):.4f}")
print("F1-score :", f"{f1_score(y_test, y_pred_balanced):.4f}")
print("Recall   :", f"{recall_score(y_test, y_pred_balanced):.4f}")
print("Precision:", f"{precision_score(y_test, y_pred_balanced):.4f}")


Метрики модели логистической регрессии (сбалансированные данные):
Accuracy : 0.9500
F1-score : 0.7317
Recall   : 0.7500
Precision: 0.7143


In [ ]:
# Балансировка данных существенно улучшила производительность модели.
# До этого, несмотря на высокую общую точность, логистическая регрессия плохо справлялась с редким классом 1 (F1-score 0.40, recall 0.35),
# правильно определяя в основном только объекты класса 0.
# После искусственного увеличения примеров редкого класса все метрики значительно выросли: точность достигла 0.95,
# F1-score увеличился до 0.73, recall – до 0.75, а precision – до 0.71.
# Модель теперь успешно распознает класс 1, сохраняя при этом высокую общую точность.
# Балансировка позволила алгоритму лучше изучить редкий класс и снизить перекос в сторону большинства.

## 2. Применение логистической регрессии (нелинейные данные)

In [133]:
x_train, y_train, x_test, y_test = load_data('dataset2')

In [134]:
# Создайте и обучите модель но этом наборе данных.
model = LogisticRegression(dim=x_train.shape[1])
model.fit(x_train, y_train)
print("Модель обучена")

Модель обучена


In [135]:
# Проанализируйте качество модели.
y_pred = model.predict(x_test)

print(f"\nМетрики модели логистической регрессии (нелинейные данные):")
print("Accuracy :", f"{accuracy_score(y_test, y_pred):.4f}")
print("F1-score :", f"{f1_score(y_test, y_pred, zero_division=0):.4f}")
print("Recall   :", f"{recall_score(y_test, y_pred, zero_division=0):.4f}")
print("Precision:", f"{precision_score(y_test, y_pred, zero_division=0):.4f}")


Метрики модели логистической регрессии (нелинейные данные):
Accuracy : 0.5700
F1-score : 0.6195
Recall   : 0.7778
Precision: 0.5147


In [136]:
# FEATURE ENGINEERING: попробуйте применить на исходных данных разные нелинейные функции (sin, tanh, ...).
# Объедините трансформированные данные с исходными (важно: количество экземпляров в x_train не должно увеличиться).

x_train_sin = np.sin(x_train)
x_train_tanh = np.tanh(x_train)
x_train_squared = x_train**2
x_train_exp = np.exp(x_train)

# Объединение:
# Используем np.hstack для объединения по горизонтали (добавление новых столбцов-признаков)
x_train_transformed = np.hstack((x_train, x_train_sin, x_train_tanh, x_train_squared, x_train_exp))

# То же самое для тестовых данных:
x_test_sin = np.sin(x_test)
x_test_tanh = np.tanh(x_test)
x_test_squared = x_test**2
x_test_exp = np.exp(x_test)

x_test_transformed = np.hstack((x_test, x_test_sin, x_test_tanh, x_test_squared, x_test_exp))

In [137]:
# Создайте и обучите модель с использованием наработок.
model = LogisticRegression(dim=x_train_transformed.shape[1])
model.fit(x_train_transformed, y_train)
print("Модель обучена")

Модель обучена


In [138]:
# Оцените качество новой модели, используя метрики из пакета sklearn.metrics.
# Указание: постарайтесь добиться точности в 100%!
y_pred_transformed = model.predict(x_test_transformed)

print(f"\nМетрики модели логистической регрессии (Feature Engineering):")
print("Accuracy :", f"{accuracy_score(y_test, y_pred_transformed):.4f}")
print("F1-score :", f"{f1_score(y_test, y_pred_transformed, zero_division=0):.4f}")
print("Recall   :", f"{recall_score(y_test, y_pred_transformed, zero_division=0):.4f}")
print("Precision:", f"{precision_score(y_test, y_pred_transformed, zero_division=0):.4f}")


Метрики модели логистической регрессии (Feature Engineering):
Accuracy : 1.0000
F1-score : 1.0000
Recall   : 1.0000
Precision: 1.0000


In [ ]:
# Анализ показал, что базовая модель, использующая исходные признаки, демонстрировала недостаточную эффективность.
# Метрики указывают на то, что модель частично справляется с задачей, но склонна к ошибкам, особенно при классификации одного из классов.
# Высокое значение Recall при низкой Precision свидетельствует о тенденции модели к ложноположительным предсказаниям.
# Для решения этой проблемы были применены нелинейные преобразования признаков.
# Эта модификация привела к значительному улучшению качества модели: все ключевые метрики (Accuracy, F1, Recall, Precision) достигли абсолютного значения 1.0.
# Это говорит о том, что расширенное признаковое пространство позволило модели идеально разделить классы.
# Ключевая причина такого успеха — устранение нелинейности границы между классами, что стало возможным благодаря введению новых, нелинейных признаков.

## 3. Доп. задания (любое на выбор, опционально)

### 3.1 'Упрощение' логистической регрессии

Сложность: легко.

In [130]:
"""
Модифицируйте класс логистической регрессии так, чтобы в нём не использовалась сигмоида.
То есть вывод о предсказанном классе должен делаться на основе значений "до сигмоиды".
Вспомогательная ссылка: https://en.wikipedia.org/wiki/Logit
"""

class LogisticRegressionModified:
    def __init__(self, dim=2):
        self.w = np.random.randn(dim, 1) / np.sqrt(dim)
        self.b = np.zeros((1,))

    def predict(self, x):
        """
        Выполняет предсказание без использования сигмоиды,
        возвращая бинарные предсказания (0 или 1) на основе z > 0.
        """
        z = x.dot(self.w) + self.b
        return np.array(z > 0).astype('int32')

In [131]:
# Перенесите обученные веса модели из пункта 1.3 в новую модель с модифицированным кодом
print("Создание модифицированной модели и перенос весов")
model_modified = LogisticRegressionModified(dim=x_train_balanced.shape[1])

# Переносим обученные веса и смещение из оригинальной модели
model_modified.w = model_balanced.w
model_modified.b = model_balanced.b

Создание модифицированной модели и перенос весов


In [132]:
# Убедитесь, что предсказания модели с модифицированными кодом совпадают с предсказаниями
# модели из пункта 1.3
# Получаем предсказания от оригинальной модели (из пункта 1.3)
y_pred_balanced_original = model_balanced.predict(x_test)

# Получаем предсказания от модифицированной модели
y_pred_modified = model_modified.predict(x_test)

# Сравниваем предсказания
if np.array_equal(y_pred_balanced_original, y_pred_modified):
    print("Предсказания модифицированной модели полностью совпадают с предсказаниями оригинальной модели.")
else:
    print("Предсказания модифицированной и оригинальной моделей НЕ совпадают.")
    diff_count = np.sum(y_pred_balanced_original != y_pred_modified)
    print(f"Количество несовпадающих предсказаний: {diff_count} из {len(y_test)}")

# Метрики для модифицированной модели, чтобы убедиться в идентичности
print(f"\nМетрики модифицированной модели:")
print("Accuracy :", f"{accuracy_score(y_test, y_pred_modified):.4f}")
print("F1-score :", f"{f1_score(y_test, y_pred_modified):.4f}")
print("Recall   :", f"{recall_score(y_test, y_pred_modified):.4f}")
print("Precision:", f"{precision_score(y_test, y_pred_modified):.4f}")

Предсказания модифицированной модели полностью совпадают с предсказаниями оригинальной модели.

Метрики модифицированной модели:
Accuracy : 0.9500
F1-score : 0.7317
Recall   : 0.7500
Precision: 0.7143


### 3.2 'Обобщение' логистической регрессии

Напишите многоклассовый классификатор. Обучите его на наборе данных ниже.

In [ ]:
x_train, y_train, x_test, y_test = load_data('dataset3')

<b>Ансамбль логистических регрессий.</b> Сложность: супергерой.

In [ ]:
"""
Напишите класс, что инкапсулирует в себе `C` логистических регрессий,
где `C` - количество классов. i-ая логистическая регрессия производит
бинарную классификацию вида: все остальные классы и i-ый класс.
"""

class MulticlassLogisticRegression:
    def __init__(self, n_classes, dim):
        pass

    def predict(self, x):
        # x - numpy массив размерности [N, dim]
        # Возвращается массив целых чисел размерности [N],
        # где i-ый элемент обозначает номер класса для
        # i-го экземпляра данных в `x`.
        pass

    def fit(self, x, y):
        pass

In [ ]:
# Создайте и обучите написанный классификатор. Оцените точность модели.


<b>Softmax классификатор.</b> Сложность: математический гений.

In [ ]:
"""
Напишите класс классификатора, основанного на функции Softmax.
Алгоритм работы данного классификатора:
x - вектор (экземпляр данных) размерности dim.
W - матрица весов размерности [dim, n_classes].

Ответ классификатора формируется как:
logits = x * W - матричное умножение
p = softmax(logits)
class_id = argmax(p)

Для данного классификатора требуется модифицировать алгоритм обучения в методе fit.

Вспомогательные ресурсы:
https://en.wikipedia.org/wiki/Softmax_function
https://eli.thegreenplace.net/2016/the-softmax-function-and-its-derivative/
"""

class SoftmaxClassificator:
    def __init__(self, n_classes, dim):
        pass

    def predict(self, x):
        # x - numpy массив размерности [N, dim]
        # Возвращается массив целых чисел размерности [N],
        # где i-ый элемент обозначает номер класса для
        # i-го экземпляра данных в `x`.
        pass

    def fit(self, x, y):
        pass

In [ ]:
# Создайте и обучите написанный классификатор. Оцените точность модели, посчитайте матрицу ошибок (выведите её с помощью matplotlib).


In [ ]:
# Создайте и обучите написанный классификатор на наборе данных из задания 1 (опционально).
# Оцените точность модели, посчитайте матрицу ошибок (выведите её с помощью matplotlib).
